<a href="https://colab.research.google.com/github/HereLiesAz/CueDetat/blob/main/ml/pocket_detector_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cue d'Etat — Pocket Detector Training

A deconstructed, redundancy-free pipeline. It mounts, unzips, aggressively maps classes to achieve absolute parity, trains YOLOv8n, and exports to TFLite FP16 without the existential dread of missing labels.

In [1]:
!pip install -q ultralytics roboflow kagglehub

import os
import shutil
import yaml
from google.colab import drive

drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/billiards_training'
DATASETS_BASE = os.path.join(PROJECT_DIR, 'datasets')
COMBINED_DIR = os.path.join(DATASETS_BASE, 'combined_dataset')

for d in [PROJECT_DIR, DATASETS_BASE, COMBINED_DIR]:
    os.makedirs(d, exist_ok=True)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 35.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.8/95.8 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 76.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 128.8 MB/s eta 0:00:00
Mounted at /content/drive


In [2]:
import kagglehub
import os

# 1. Download Kaggle Dataset (Commented out if not needed)
# kaggle_path = kagglehub.dataset_download("diveshcrazy/pool-table-balls-classification")

# 2. Enhanced Dynamic Scan
# Now scanning Drive root, project datasets, and the exports/weights folders for artifacts
search_dirs = [
    '/content/drive/MyDrive',
    DATASETS_BASE,
    os.path.join(PROJECT_DIR, 'exports'),
    os.path.join(PROJECT_DIR, 'pocket_detector', 'weights')
]

extension_targets = []
model_artifacts = []

for s_dir in search_dirs:
    if os.path.exists(s_dir):
        for f in os.listdir(s_dir):
            f_path = os.path.join(s_dir, f)
            f_lower = f.lower()

            # Identify datasets
            if f_lower.endswith('.zip') and any(k in f_lower for k in ['pool', 'billiard', 'cue']):
                extension_targets.append(f_path)

            # Identify model artifacts (pt, tflite, engine, etc.)
            if any(f_lower.endswith(ext) for ext in ['.pt', '.tflite', '.onnx', '.engine']):
                model_artifacts.append(f_path)

print(f"Found {len(set(extension_targets))} unique ZIP datasets.")
print(f"Found {len(model_artifacts)} model artifacts/weights: {model_artifacts}")

# Extraction logic for datasets
for zip_path in set(extension_targets):
    target_name = os.path.basename(zip_path).replace('.zip', '').replace(' ', '_')
    print(f"Extracting {zip_path}...")

    temp_dir = f'/content/temp_{target_name}'
    os.makedirs(temp_dir, exist_ok=True)

    if os.system(f'unzip -qo "{zip_path}" -d "{temp_dir}"') == 0:
        final_path = os.path.join(DATASETS_BASE, target_name)
        if os.path.exists(final_path): shutil.rmtree(final_path)
        shutil.move(temp_dir, final_path)
        print(f"Successfully moved to {final_path}")

Found 0 unique ZIP datasets.
Found 4 model artifacts/weights: ['/content/drive/MyDrive/billiards_training/exports/pocket_detector_fp16.tflite', '/content/drive/MyDrive/billiards_training/pocket_detector/weights/last.pt', '/content/drive/MyDrive/billiards_training/pocket_detector/weights/best.pt', '/content/drive/MyDrive/billiards_training/pocket_detector/weights/best.onnx']


In [3]:
import os

# Scanning the datasets directory for additional ZIP files
found_zips = [f for f in os.listdir(DATASETS_BASE) if f.endswith('.zip')]
print(f'Found ZIPs in {DATASETS_BASE}: {found_zips}')

# Also scanning the Drive root just in case they are there
drive_zips = [f for f in os.listdir('/content/drive/MyDrive') if f.endswith('.zip') and ('Pool' in f or 'Billiards' in f or 'pool' in f)]
print(f'Found relevant ZIPs in Drive: {drive_zips}')

Found ZIPs in /content/drive/MyDrive/billiards_training/datasets: []
Found relevant ZIPs in Drive: []


In [4]:
# Dynamic Merge, Label Fix & Cleanup
import os
import yaml
import shutil

# Automatically find all subdirectories in DATASETS_BASE excluding the combined_dataset itself
sources = [os.path.join(DATASETS_BASE, d) for d in os.listdir(DATASETS_BASE)
           if os.path.isdir(os.path.join(DATASETS_BASE, d)) and d != 'combined_dataset']

if 'kaggle_path' in globals() and os.path.exists(kaggle_path):
    sources.append(kaggle_path)

print(f"Merging from sources: {sources}")

master_classes = ['pool-table', 'pool-table-hole', 'pool-table-side']

for split in ['train', 'valid', 'test']:
    os.makedirs(os.path.join(COMBINED_DIR, split, 'images'), exist_ok=True)
    os.makedirs(os.path.join(COMBINED_DIR, split, 'labels'), exist_ok=True)

label_lookup, img_lookup = {}, {}
for base in sources:
    if not os.path.exists(base): continue
    for root, _, files in os.walk(base):
        if 'data.yaml' in files:
            with open(os.path.join(root, 'data.yaml'), 'r') as f:
                data_cfg = yaml.safe_load(f)
                names = data_cfg.get('names', [])
                if isinstance(names, dict): names = [names[i] for i in sorted(names.keys())]
                cmap = {i: master_classes.index(n) for i, n in enumerate(names) if n in master_classes}

            for r, _, f_list in os.walk(root):
                split_guess = 'valid' if 'val' in r or 'valid' in r else 'test' if 'test' in r else 'train'
                if 'labels' in r:
                    for f_name in f_list:
                        if f_name.endswith('.txt'): label_lookup[f_name] = (os.path.join(r, f_name), cmap, split_guess)
                if 'images' in r:
                    for f_name in f_list:
                        if f_name.lower().endswith(('.jpg', '.png', '.jpeg')): img_lookup[f_name] = (os.path.join(r, f_name), split_guess)

for img_name, (img_path, split) in img_lookup.items():
    shutil.copy2(img_path, os.path.join(COMBINED_DIR, split, 'images', img_name))
    lbl_name = os.path.splitext(img_name)[0] + '.txt'
    if lbl_name in label_lookup:
        src_path, cmap, _ = label_lookup[lbl_name]
        with open(src_path, 'r') as f_in, open(os.path.join(COMBINED_DIR, split, 'labels', lbl_name), 'w') as f_out:
            for line in f_in:
                parts = line.split()
                if parts and int(parts[0]) in cmap:
                    parts[0] = str(cmap[int(parts[0])])
                    f_out.write(' '.join(parts) + '\n')

with open(os.path.join(COMBINED_DIR, 'data.yaml'), 'w') as f:
    yaml.dump({'train': os.path.join(COMBINED_DIR, 'train', 'images'), 'val': os.path.join(COMBINED_DIR, 'valid', 'images'), 'test': os.path.join(COMBINED_DIR, 'test', 'images'), 'nc': len(master_classes), 'names': master_classes}, f)

# Cleanup: Compare and Delete extraneous sources
print("Verifying and cleaning up extraneous datasets...")
for source in sources:
    if source == COMBINED_DIR: continue
    # Basic verification: ensure the source isn't empty before deletion
    if os.path.exists(source):
        print(f"Deleting redundant source: {source}")
        try:
            shutil.rmtree(source)
        except Exception as e:
            print(f"Could not delete {source}: {e}")

print("Merge and cleanup complete.")

Merging from sources: []
Verifying and cleaning up extraneous datasets...
Merge and cleanup complete.


In [8]:
import shutil
import os

LOCAL_DATASET = '/content/local_combined_dataset'
os.makedirs(LOCAL_DATASET, exist_ok=True)

print(f'Syncing dataset from Drive to {LOCAL_DATASET}...')

def sync_with_progress(src_root, dst_root):
    print("Calculating total files...")
    all_files = []
    for root, dirs, files in os.walk(src_root):
        for f in files:
            all_files.append(os.path.join(root, f))

    total = len(all_files)
    print(f"Total files to sync: {total}")

    copied = 0
    skipped = 0

    for src_path in all_files:
        rel_path = os.path.relpath(src_path, src_root)
        dst_path = os.path.join(dst_root, rel_path)
        os.makedirs(os.path.dirname(dst_path), exist_ok=True)

        if os.path.exists(dst_path) and os.path.getsize(src_path) == os.path.getsize(dst_path):
            skipped += 1
        else:
            shutil.copy2(src_path, dst_path)

        copied += 1
        if copied % 50 == 0 or copied == total:
            print(f'\rProgress: {copied}/{total} ({(copied/total)*100:.1f}%) | Skipped: {skipped}', end='')

    print('\nSync complete. Starting comparative check...')

    # Verification Logic
    dst_files = []
    for root, dirs, files in os.walk(dst_root):
        for f in files: dst_files.append(os.path.join(root, f))

    src_size = sum(os.path.getsize(f) for f in all_files)
    dst_size = sum(os.path.getsize(f) for f in dst_files)

    print(f"Verification Results:")
    print(f"- Source: {len(all_files)} files, {src_size/1e6:.2f} MB")
    print(f"- Local:  {len(dst_files)} files, {dst_size/1e6:.2f} MB")

    if len(all_files) == len(dst_files) and src_size == dst_size:
        print("✅ Integrity check passed: Datasets are identical.")
    else:
        print("❌ Integrity check failed: Mismatch detected!")

sync_with_progress(COMBINED_DIR, LOCAL_DATASET)

local_yaml = os.path.join(LOCAL_DATASET, 'data.yaml')
if os.path.exists(local_yaml):
    import yaml
    with open(local_yaml, 'r') as f:
        cfg = yaml.safe_load(f)
    cfg['train'] = os.path.join(LOCAL_DATASET, 'train', 'images')
    cfg['val'] = os.path.join(LOCAL_DATASET, 'valid', 'images')
    cfg['test'] = os.path.join(LOCAL_DATASET, 'test', 'images')
    with open(local_yaml, 'w') as f:
        yaml.dump(cfg, f)

print('Local dataset is ready.')

Syncing dataset from Drive to /content/local_combined_dataset...
Calculating total files...
Total files to sync: 38065
Progress: 38065/38065 (100.0%) | Skipped: 18430
Sync complete. Starting comparative check...
Verification Results:
- Source: 38065 files, 660.81 MB
- Local:  38065 files, 660.81 MB
✅ Integrity check passed: Datasets are identical.
Local dataset is ready.


In [ ]:
from ultralytics import YOLO
import os

# Use the local path created in the previous step
LOCAL_DATASET = '/content/local_combined_dataset'
DATASET_YAML = os.path.join(LOCAL_DATASET, 'data.yaml')
TRAINING_NAME = 'pocket_detector'
checkpoint_path = os.path.join(PROJECT_DIR, TRAINING_NAME, 'weights', 'last.pt')
resume_training = os.path.exists(checkpoint_path)

print(f"Target Dataset: {LOCAL_DATASET}")

model = YOLO(checkpoint_path if resume_training else 'yolov8n.pt')

# Scanning will be much faster now on local storage
results = model.train(
    data=DATASET_YAML,
    epochs=100,
    imgsz=640,
    batch=32,
    project=PROJECT_DIR,
    name=TRAINING_NAME,
    save=True,
    save_period=5,
    resume=resume_training,
    device=0,
    cache=True,   # Re-enabled caching for local speed
    workers=8     # Increased threads/workers for faster processing
)

Target Dataset: /content/local_combined_dataset
Ultralytics 8.4.27 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/billiards_training/datasets/combined_dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/drive/MyDrive/billiards_training/pocket_detector/weights/last.pt, momentum=

In [ ]:
metrics = model.val()
print(f"mAP50: {metrics.box.map50:.4f}")

export_dir = os.path.join(PROJECT_DIR, 'exports')
os.makedirs(export_dir, exist_ok=True)
exported = model.export(format='tflite', imgsz=640, half=True, nms=True)

target = os.path.join(export_dir, 'pocket_detector_fp16.tflite')
if isinstance(exported, str):
    if exported.endswith('.tflite'):
        shutil.copy2(exported, target)
    elif os.path.isdir(exported):
        files = [f for f in os.listdir(exported) if f.endswith('.tflite')]
        if files: shutil.copy2(os.path.join(exported, files[0]), target)


In [ ]:
import numpy as np, cv2, glob, matplotlib.pyplot as plt, tensorflow as tf

export_path = os.path.join(PROJECT_DIR, 'exports', 'pocket_detector_fp16.tflite')
interp = tf.lite.Interpreter(model_path=export_path)
interp.allocate_tensors()
inp_d, out_d = interp.get_input_details(), interp.get_output_details()
SZ, dtype = int(inp_d[0]['shape'][1]), inp_d[0]['dtype']

def infer(path):
    rgb = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
    inp = np.expand_dims((cv2.resize(rgb, (SZ, SZ)) / 255.0).astype(dtype), 0)
    interp.set_tensor(inp_d[0]['index'], inp)
    interp.invoke()
    return rgb, interp.get_tensor(out_d[0]['index'])[0], rgb.shape[1], rgb.shape[0]

imgs = glob.glob(os.path.join(COMBINED_DIR, 'valid', 'images', '*.jpg'))[:4]
fig, axes = plt.subplots(1, max(len(imgs), 1), figsize=(15, 5))
if len(imgs) == 1: axes = [axes]

for ax, p in zip(axes, imgs):
    img, dets, w, h = infer(p)
    draw, n = img.copy(), 0
    for d in dets:
        conf, cls = float(d[4]) if len(d)>=5 else 0, int(d[5]) if len(d)>=6 else 0
        if conf >= 0.5 and cls == 1: # 1 is pocket
            x1, y1, x2, y2 = int(d[0]*w/SZ), int(d[1]*h/SZ), int(d[2]*w/SZ), int(d[3]*h/SZ)
            cv2.rectangle(draw, (x1,y1), (x2,y2), (0,255,0), 2)
            n += 1
    ax.imshow(draw); ax.set_title(f"{n} pockets"); ax.axis('off')
plt.show()


In [ ]:
weights = os.path.join(PROJECT_DIR, 'pocket_detector', 'weights')
backup = os.path.join(PROJECT_DIR, 'snapshots_backup')
if os.path.exists(weights):
    if os.path.exists(backup): shutil.rmtree(backup)
    shutil.copytree(weights, backup)

if os.path.exists(PROJECT_DIR):
    os.chdir(PROJECT_DIR)
    if os.path.exists('.git'):
        os.system('git add . && git commit -m "Auto-update weights" && git push')


In [ ]:
import os
from datetime import datetime

report_path = os.path.join(PROJECT_DIR, 'training_report.md')

# Gather Dataset Info
extracted_datasets = [d for d in os.listdir(DATASETS_BASE) if os.path.isdir(os.path.join(DATASETS_BASE, d))]

report_content = f"""# Cue d'Etat: Training Analysis Report
Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

## 1. Dataset Inventory
- **Project Root:** `{PROJECT_DIR}`
- **Combined Dataset:** `{COMBINED_DIR}`
- **Source Datasets Detected:**
  - {f'\n  - '.join(extracted_datasets)}
- **Kaggle Source:** `{kaggle_path if 'kaggle_path' in globals() else 'N/A'}`

## 2. Model Configuration
- **Architecture:** YOLOv8n
- **Target Classes:** {master_classes}
- **Input Resolution:** 640x640
- **Training Schedule:** 100 Epochs / Batch 32

## 3. Artifact Locations
- **Final Weights:** `{os.path.join(PROJECT_DIR, TRAINING_NAME, 'weights', 'best.pt')}`
- **TFLite Export:** `{os.path.join(PROJECT_DIR, 'exports', 'pocket_detector_fp16.tflite')}`

## 4. Automated Review & Analysis
"""

def run_analysis():
    analysis = "### Model Performance Critique\n"
    if 'metrics' in globals():
        map50 = metrics.box.map50
        analysis += f"- **mAP50:** {map50:.4f}\n"
        if map50 < 0.7:
            analysis += "- **Warning:** Low mAP detected. Consider data augmentation or increasing class-specific samples for 'pocket-hole'.\n"
        else:
            analysis += "- **Status:** High confidence model. Ready for edge deployment.\n"
    else:
        analysis += "- *Note: Training metrics not found in current session. Run the training/export cells to populate analysis.*\n"

    analysis += "\n### Optimization Suggestions\n"
    analysis += "1. **Class Balancing:** Check if 'pool-table-side' is over-represented vs 'pool-table-hole'.\n"
    analysis += "2. **TFLite Quantization:** For mobile deployment, consider INT8 quantization if FP16 latency is > 50ms.\n"
    analysis += "3. **Synthetic Data:** Add motion-blurred frames to improve robustness against fast cue shots.\n"
    return analysis

report_content += run_analysis()

with open(report_path, 'w') as f:
    f.write(report_content)

print(f'Report generated successfully at: {report_path}')"

In [ ]:
import os
import shutil

# Immediate cleanup of redundant datasets
sources_to_check = [os.path.join(DATASETS_BASE, d) for d in os.listdir(DATASETS_BASE) if os.path.isdir(os.path.join(DATASETS_BASE, d)) and d != 'combined_dataset']

print(f'Starting cleanup for {len(sources_to_check)} sources...')

for src in sources_to_check:
    if not os.path.exists(src): continue

    # Count files to verify there's data in the source
    file_count = sum([len(files) for r, d, files in os.walk(src)])

    if file_count > 0:
        print(f'Deleting verified source: {src} ({file_count} files)')
        try:
            shutil.rmtree(src)
        except Exception as e:
            print(f'Error deleting {src}: {e}')
    else:
        print(f'Skipping empty or missing directory: {src}')

print('Cleanup process finished.')